In [4]:
import timeit
import cvxpy as cp
import numpy as np

The idea is to test the performance of the QP vs SOCP on the Ridge part of Elastic Net.

In [34]:
n,m = 1000, 100


def random_socp_call():
    lambda_2 = np.random.randint(0,1000)
    psi_X_i = lambda_2 ** 2 -  np.random.rand()

    # Subproblem i with b_i, z_i, u_i
    b_i = cp.Variable(1, name="beta_i")
    u_i = cp.Variable(1, name="u_i")
    aux = cp.Variable(2, name="aux")

    # Constraints
    constraints = [
        aux[0] == 2*b_i,
        aux[1] == u_i - 1,
        u_i >= 0
    ]

    constraints += [
        cp.SOC(
            u_i + 1,
            aux 
        )
    ]

    # Problem
    socp = cp.Problem(
        cp.Minimize(
            psi_X_i * b_i + lambda_2 * u_i
        ),
        constraints
        )

    try:
        socp.solve(
            verbose=False, 
            solver=cp.MOSEK, 
            )
    except Exception as e:
        print(e)  


def random_qp_call():
    lambda_2 = np.random.randint(0,1000)
    psi_X_i = lambda_2 ** 2 -  np.random.rand()
    
    # Subproblem i with b_i, z_i, u_i
    b_i = cp.Variable(1, name="beta_i")

    # Problem
    socp = cp.Problem(
        cp.Minimize(
            psi_X_i * b_i + lambda_2 * b_i ** 2
        ),
        # constraints
        )

    try:
        socp.solve(
            verbose=False, 
            solver=cp.MOSEK, 
            )
    except Exception as e:
        print(e)  


In [35]:
print("Benchmarking SOCP...")
%timeit random_socp_call()
print("Benchmarking QP...")
%timeit random_qp_call()

Benchmarking SOCP...
Solver 'MOSEK' failed. Try another solver, or solve with verbose=True for more information.
14.3 ms ± 186 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Benchmarking QP...
Solver 'MOSEK' failed. Try another solver, or solve with verbose=True for more information.
Solver 'MOSEK' failed. Try another solver, or solve with verbose=True for more information.
10.8 ms ± 189 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
